<a href="https://colab.research.google.com/github/xingji1337/week5RAG/blob/TrackC-LightweightEvalWithGuardrails/Week5_3_RAG_Eval_Guardrails_HOME_REPAIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 — Track C: RAG Evaluation & Guardrails (Home Repair AI)

This notebook builds an **evaluation and guardrail framework** for the *Home Repair Assistant* RAG system.

It will:
- Load a small evaluation set of home-repair queries (with expected supporting docs).
- Run the retrieval pipeline (baseline vs advanced).
- Compute automatic metrics (hit-rate, context recall proxy).
- Log latency and context size.
- Add guardrail checks (citation enforcement, safe refusal).
- Export results for your report.


## 0) Install dependencies

In [2]:
!pip install sentence-transformers rank_bm25 faiss-cpu pypdf rapidfuzz pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.2 MB/s eta 0:00:00


## 1) Set paths & load eval set

In [3]:
import os, sys, time, json, math, random, pathlib, re
from typing import List, Dict, Any, Tuple

# If using Colab and your PDFs live in Drive, mount then point DATA_DIR to your folder.
USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Drive mounted.")
    except Exception as e:
        print("Colab not detected or mount failed:", e)

# Default to the uploaded files path in this environment:
# DATA_DIR = '/mnt/data' # Original path

# Update DATA_DIR to point to the correct directory containing the uploaded PDF files
DATA_DIR = '/content/drive/MyDrive/' # Assuming the PDFs are in the default Colab content directory

PDF_FILES = [
    os.path.join(DATA_DIR, 'Complete home repair  with 350 projects and 2300 photos.pdf'),
    os.path.join(DATA_DIR, '7 Different Ways to Repair Drywall.pdf'),
    os.path.join(DATA_DIR, 'How to stop Water damage when A Leak.pdf')
]

for p in PDF_FILES:
    print("Exists?", os.path.exists(p), p)

Mounted at /content/drive
Drive mounted.
Exists? True /content/drive/MyDrive/Complete home repair  with 350 projects and 2300 photos.pdf
Exists? True /content/drive/MyDrive/7 Different Ways to Repair Drywall.pdf
Exists? True /content/drive/MyDrive/How to stop Water damage when A Leak.pdf


## 2) Load corpus (reuse from Track A)

In [5]:
## 2) Load corpus (self-contained version, reusing Track A functions)

import os, pathlib, re
from dataclasses import dataclass
from typing import List, Dict, Any
from pypdf import PdfReader

# Paths to your uploaded home-repair PDFs
# DATA_DIR = '/mnt/data' # Original path
DATA_DIR = '/content/drive/MyDrive/' # Updated path

PDF_FILES = [
    os.path.join(DATA_DIR, 'Complete home repair  with 350 projects and 2300 photos.pdf'),
    os.path.join(DATA_DIR, '7 Different Ways to Repair Drywall.pdf'),
    os.path.join(DATA_DIR, 'How to stop Water damage when A Leak.pdf')
]

# ---- Text loader + chunker ----
def load_pdf_text(path: str) -> List[str]:
    pages = []
    reader = PdfReader(path)
    for i, page in enumerate(reader.pages):
        txt = page.extract_text() or ""
        if txt.strip():
            pages.append(txt)
    return pages

def chunk_text(text: str, chunk_size: int = 800, chunk_overlap: int = 150) -> List[str]:
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i+chunk_size]
        chunks.append(" ".join(chunk))
        i += (chunk_size - chunk_overlap)
    return chunks

@dataclass
class DocChunk:
    doc_id: str
    chunk_id: int
    text: str
    source: str
    meta: Dict[str, Any]

def build_corpus(pdf_files: List[str], chunk_size=800, chunk_overlap=150) -> List[DocChunk]:
    corpus = []
    for src in pdf_files:
        doc_id = pathlib.Path(src).stem
        pages = load_pdf_text(src)
        full = "\n\n".join(pages)
        chunks = chunk_text(full, chunk_size, chunk_overlap)
        for idx, ch in enumerate(chunks):
            corpus.append(DocChunk(
                doc_id=doc_id,
                chunk_id=idx,
                text=ch,
                source=src,
                meta={"doc": doc_id, "chunk": idx}
            ))
    print(f"Built corpus with {len(corpus)} chunks from {len(pdf_files)} PDFs.")
    return corpus

# Build it
corpus = build_corpus(PDF_FILES)

Built corpus with 348 chunks from 3 PDFs.


In [14]:
from rank_bm25 import BM25Okapi

tokenized = [c.text.split() for c in corpus]
bm25 = BM25Okapi(tokenized)

def bm25_search(query: str, k: int = 20) -> List[Tuple[float, DocChunk]]:
    scores = bm25.get_scores(query.split())
    ranked = sorted(list(enumerate(scores)), key=lambda x: x[1], reverse=True)[:k]
    return [(score, corpus[idx]) for idx, score in ranked]


In [15]:
try:
    from sentence_transformers import SentenceTransformer, util
    _ST_AVAILABLE = True
except Exception as e:
    _ST_AVAILABLE = False
    print("sentence-transformers not available; install first to use dense retrieval.")

dense_model_name = "sentence-transformers/all-MiniLM-L6-v2"  # fast & light
dense_model = None
dense_embeddings = None

def ensure_dense():
    global dense_model, dense_embeddings
    if not _ST_AVAILABLE:
        raise RuntimeError("Install sentence-transformers to enable dense retrieval.")
    if dense_model is None:
        dense_model = SentenceTransformer(dense_model_name)
        texts = [c.text for c in corpus]
        dense_embeddings = dense_model.encode(texts, convert_to_tensor=True, show_progress_bar=True)
        print("Dense index built.")
    return dense_model, dense_embeddings

def dense_search(query: str, k: int = 20) -> List[Tuple[float, DocChunk]]:
    model, embs = ensure_dense()
    q = model.encode([query], convert_to_tensor=True)
    cos = util.cos_sim(q, embs)[0]
    topk = int(min(k, len(corpus)))
    vals, idxs = cos.topk(topk)
    out = []
    for score, idx in zip(vals.tolist(), idxs.tolist()):
        out.append((float(score), corpus[idx]))
    return out

In [16]:
from collections import defaultdict

def rrf_fuse(results_lists: List[List[DocChunk]], K: float = 60.0, topk: int = 20) -> List[Tuple[float, DocChunk]]:
    # results_lists: list of ranked lists [(score, chunk), ...]
    rrfs = defaultdict(float)
    ranks = defaultdict(dict)

    for rl_idx, rl in enumerate(results_lists):
        for rank, (score, ch) in enumerate(rl, start=1):
            key = (ch.doc_id, ch.chunk_id)
            rrfs[key] += 1.0 / (K + rank)
            ranks[key] = ch

    fused = sorted([(score, ranks[key]) for key, score in rrfs.items()],
                   key=lambda x: x[0], reverse=True)[:topk]
    return fused

def retrieve_rrf(query: str, k_each: int = 20, topk: int = 20) -> List[Tuple[float, DocChunk]]:
    b = bm25_search(query, k=k_each)
    try:
        d = dense_search(query, k=k_each)
    except Exception:
        d = []
    return rrf_fuse([b, d], K=60.0, topk=topk)

In [22]:
import numpy as np
def mmr(query: str, ranked: List[Tuple[float, DocChunk]], lambda_mult: float = 0.7, topk: int = 10):
    # requires dense model to compute repulsions; if absent, returns topk
    if not _ST_AVAILABLE:
        return ranked[:topk]
    model, _ = ensure_dense()
    q_vec = model.encode([query], convert_to_tensor=False, normalize_embeddings=True)[0]
    cand_texts = [ch.text for _, ch in ranked]
    cand_vecs = model.encode(cand_texts, convert_to_tensor=False, normalize_embeddings=True)

    selected, selected_idx = [], []
    remaining = list(range(len(cand_vecs)))

    while remaining and len(selected) < topk:
        best_idx, best_score = None, -1e9
        for i in remaining:
            rel = np.dot(q_vec, cand_vecs[i])
            div = 0.0 if not selected_idx else max(np.dot(cand_vecs[i], cand_vecs[j]) for j in selected_idx)
            score = lambda_mult * rel - (1 - lambda_mult) * div
            if score > best_score:
                best_score, best_idx = score, i
        selected_idx.append(best_idx)
        remaining.remove(best_idx)
        selected.append(ranked[best_idx])
    return selected

In [24]:
# Simple heuristic compressor: keep top-N sentences v2 using RapidFuzz for sentence scoring
from rapidfuzz import fuzz

def compress_chunk(query: str, text: str, max_chars: int = 900) -> str:
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    scored = [(fuzz.partial_ratio(query, s), s) for s in sents]
    scored.sort(key=lambda x: x[0], reverse=True)
    out = ""
    for sc, s in scored:
        if len(out) + len(s) + 1 > max_chars:
            break
        out += (" " if out else "") + s
    return out if out else text[:max_chars]

def build_context(query: str, ranked: List[Tuple[float, DocChunk]], max_context_chars: int = 3500):
    ctx_parts = []
    total = 0
    for _, ch in ranked:
        comp = compress_chunk(query, ch.text, max_chars=900)
        tag = f"[{ch.doc_id}#${ch.chunk_id}]"
        seg = f"{tag} {comp}"
        if total + len(seg) + 2 > max_context_chars:
            break
        ctx_parts.append(seg)
        total += len(seg) + 2
    return "\n\n".join(ctx_parts)


In [26]:
# For portability, we won't call a remote LLM here. We simulate formatting.
def answer_with_context(query: str, context: str) -> str:
    # In production: call your LLM with a prompt that *requires* inline citations like [doc_id#$chunk].
    return f"""Q: {query}

Relevant context (truncated):
{context[:700]}

A (draft): Based on the retrieved context above, here is a grounded answer with inline citations.
"""

def run_pipeline(query: str, k_each=20, topk=15, lambda_mmr=0.7):
    t0 = time.time()
    fused = retrieve_rrf(query, k_each=k_each, topk=topk)
    fused = rerank(query, fused, topk=topk)
    fused = mmr(query, fused, lambda_mult=lambda_mmr, topk=min(10, topk))
    context = build_context(query, fused, max_context_chars=3500)
    t1 = time.time()
    ans = answer_with_context(query, context)
    logs = {
        "latency_s": round(t1 - t0, 3),
        "num_ctx_chunks": context.count('\n\n') + 1 if context else 0,
        "ctx_chars": len(context)
    }
    return ans, context, logs, fused

example_query = "How do I repair a small doorknob hole in drywall and what safety steps should I take?"
ans, ctx, logs, fused = run_pipeline(example_query)
print(ans)
print("\nLogs:", logs)


Q: How do I repair a small doorknob hole in drywall and what safety steps should I take?

Relevant context (truncated):
[7 Different Ways to Repair Drywall#$1] When dry, sand lightly, then prime and paint. Step 2: Use a four- or six-inch-wide drywall knife to apply joint compound over the patch. Corner bead is nailed over the corner and then concealed by two or three layers of joint compound. Problem 1: Doorknob Damage Step 1: One of the most common drywall repairs occurs when a door is swung open a little too forcefully and the doorknob punches a hole through the drywall. ⚒️ Problem 2: Crumpled Corner Bead Step 1: When two sheets of drywall meet at an outside wall corner, they're protected by an L-shaped metal strip called a corner bead. For smaller repairs, something like this will suffice. The good news is

A (draft): Based on the retrieved context above, here is a grounded answer with inline citations.


Logs: {'latency_s': 3.482, 'num_ctx_chunks': 3, 'ctx_chars': 2737}


## 3) Retrieval pipeline wrapper

In [28]:
def run_pipeline(query: str, k_each=20, topk=15, lambda_mmr=0.7):
    t0 = time.time()
    fused = retrieve_rrf(query, k_each=k_each, topk=topk)
    fused = rerank(query, fused, topk=topk)          # if enabled
    fused = mmr(query, fused, lambda_mult=lambda_mmr, topk=min(10, topk))
    context = build_context(query, fused, max_context_chars=3500)
    t1 = time.time()
    ans = answer_with_context(query, context)        # from Track A
    logs = {
        "latency_s": round(t1 - t0, 3),
        "num_ctx_chunks": context.count('\n\n') + 1 if context else 0,
        "ctx_chars": len(context)
    }
    return ans, context, logs


## 4) Metrics: hit rate, latency, context size

In [29]:
import pandas as pd

# Define a sample evaluation dataset
data = {'query': ['how to repair a drywall hole', 'how to fix a leaky faucet', 'how to install a new light fixture'],
        'gold_doc_id': ['7 Different Ways to Repair Drywall', 'How to stop Water damage when A Leak', 'Complete home repair  with 350 projects and 2300 photos']}
eval_df = pd.DataFrame(data)

print("Evaluation dataset loaded:")
display(eval_df)

Evaluation dataset loaded:


,query,gold_doc_id
0,how to repair a drywall hole,7 Different Ways to Repair Drywall
1,how to fix a leaky faucet,How to stop Water damage when A Leak
2,how to install a new light fixture,Complete home repair with 350 projects and 23...


In [30]:
def evaluate(eval_df):
    rows = []
    for _, row in eval_df.iterrows():
        q, gold = row['query'], row['gold_doc_id']
        ans, ctx, logs = run_pipeline(q)
        hit = gold in ctx
        rows.append({
            "query": q,
            "gold": gold,
            "hit": hit,
            "latency_s": logs["latency_s"],
            "ctx_chars": logs["ctx_chars"],
            "ans": ans
        })
    return pd.DataFrame(rows)

results = evaluate(eval_df)
results


,query,gold,hit,latency_s,ctx_chars,ans
0,how to repair a drywall hole,7 Different Ways to Repair Drywall,True,1.853,2744,Q: how to repair a drywall hole\n\nRelevant co...
1,how to fix a leaky faucet,How to stop Water damage when A Leak,False,1.773,2692,Q: how to fix a leaky faucet\n\nRelevant conte...
2,how to install a new light fixture,Complete home repair with 350 projects and 23...,True,1.756,2765,Q: how to install a new light fixture\n\nRelev...


## 5) Guardrails: citation enforcement + safe refusal

In [31]:
import re

def enforce_citations(answer: str) -> bool:
    return bool(re.search(r'\[[^\]]+#\$\d+\]', answer))

def safe_refusal(query: str, context: str) -> str:
    if not context.strip():
        return "Sorry — I could not find grounded information in the current corpus."
    return ""

# Example usage
ans = "A draft with [doc#$0] citation"
print("Has citation?", enforce_citations(ans))

no_ctx = ""
print("Safe refusal:", safe_refusal("What about solar panels?", no_ctx))


Has citation? True
Safe refusal: Sorry — I could not find grounded information in the current corpus.


## 6) Evaluate with guardrails

In [32]:
def evaluate_with_guardrails(eval_df):
    rows = []
    for _, row in eval_df.iterrows():
        q, gold = row['query'], row['gold_doc_id']
        ans, ctx, logs = run_pipeline(q)
        hit = gold in ctx
        has_cite = enforce_citations(ans)
        refusal = safe_refusal(q, ctx)
        rows.append({
            "query": q,
            "gold": gold,
            "hit": hit,
            "has_citation": has_cite,
            "refusal_triggered": bool(refusal),
            "latency_s": logs["latency_s"],
            "ctx_chars": logs["ctx_chars"],
            "ans": ans if not refusal else refusal
        })
    return pd.DataFrame(rows)

guarded_results = evaluate_with_guardrails(eval_df)
guarded_results


,query,gold,hit,has_citation,refusal_triggered,latency_s,ctx_chars,ans
0,how to repair a drywall hole,7 Different Ways to Repair Drywall,True,True,False,1.825,2744,Q: how to repair a drywall hole\n\nRelevant co...
1,how to fix a leaky faucet,How to stop Water damage when A Leak,False,True,False,1.831,2692,Q: how to fix a leaky faucet\n\nRelevant conte...
2,how to install a new light fixture,Complete home repair with 350 projects and 23...,True,True,False,1.784,2765,Q: how to install a new light fixture\n\nRelev...


## 7) Save results

In [33]:
out_dir = './week5_outputs_eval'
os.makedirs(out_dir, exist_ok=True)
guarded_results.to_csv(os.path.join(out_dir, 'eval_guardrails_results.csv'), index=False)
print("Saved results to", out_dir)


Saved results to ./week5_outputs_eval


## Notes for your report
- Track C requires comparing retrieval performance and checking if guardrails work.
- Metrics: hit rate (did gold doc appear?), citation presence, refusal on no support, latency, context size.
- Include at least one failure case (query outside corpus) to show refusal behavior.
